In [1]:
import torch
torch.cuda.empty_cache()

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import numpy as np
from tqdm.auto import tqdm
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

# Add parent directories to path
# Navigate from train_notebook/finetuned-XLSR/ to train/
script_dir = Path.cwd().parent.parent
sys.path.insert(0, str(script_dir))

from config import (
    PROJECT_ROOT,
    SAMPLING_RATE,
    XLSR_MAX_TIME_STEPS
)
from model import SSLModel, AD_XLSR_Model
from data_split import create_train_val_split
from visualization import plot_training_curves, plot_dataset_comparison


ModuleNotFoundError: No module named 'config'

In [ ]:
DATASET_NAME = "Pitt"
FEATURE_TYPE = "xlsr_finetune"
DENOISE_METHOD = "mossformer"

# 使用PROJECT_ROOT构建路径
RAW_AUDIO_DIR = PROJECT_ROOT / f"data/denoised/{DATASET_NAME}-{DENOISE_METHOD}"
TRAIN_CSV = PROJECT_ROOT / f"data/processed/{DATASET_NAME}-{DENOISE_METHOD}-xlsr-train.csv"
VAL_CSV = PROJECT_ROOT / f"data/processed/{DATASET_NAME}-{DENOISE_METHOD}-xlsr-val.csv"
XLSR_FEATURES_DIR = PROJECT_ROOT / f"data/processed/{DATASET_NAME}_{DENOISE_METHOD}_xlsr_features"
FEATURE_DIR_NAME = f"{DATASET_NAME}_{DENOISE_METHOD}_xlsr_features"
MODEL_OUTPUT_DIR = PROJECT_ROOT / f"models/{DATASET_NAME}-{DENOISE_METHOD}_{FEATURE_TYPE}_multi_seed"

In [ ]:
AUDIO_LENGTH_SEC = 60
FINETUNE_BATCH_SIZE = 4
ACCUMULATION_STEPS = 4
XLSR_LR = 1e-5
CLASSIFIER_LR = 1e-3
FINETUNE_MAX_EPOCHS = 50
FINETUNE_PATIENCE = 10
NUM_WORKERS = 0

FINETUNE_LAYERS_NUM = 3
CLASS_WEIGHT_CONTROL = 1.5
CLASS_WEIGHT_DEMENTIA = 1.0

XLSR_DROPOUT = 0.4    
WEIGHT_DECAY = 5e-2

## Device Setup


In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    accelerator = 'gpu'
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    accelerator = 'mps'
else:
    device = torch.device('cpu')
    accelerator = 'cpu'
    
print(f"Using device: {device}")

## Step 1: Data Preparation

Create train/validation split using existing CSV files


In [ ]:
TRAIN_CSV, VAL_CSV = create_train_val_split(
    raw_audio_dir=RAW_AUDIO_DIR,
    train_csv_path=TRAIN_CSV,
    val_csv_path=VAL_CSV,
    feature_dir_name=FEATURE_DIR_NAME,
    dataset_name=f"{DATASET_NAME}-{DENOISE_METHOD}",
    xlsr=True
)

In [ ]:
def train_one_epoch(model, train_loader, optimizer, device, epoch=None, accumulation_steps=1, class_weights=None):
    """Train for one epoch with gradient accumulation"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    desc = f"Epoch {epoch} - Training" if epoch is not None else "Training"
    pbar = tqdm(train_loader, desc=desc, leave=False)
    
    optimizer.zero_grad()
    
    for batch_idx, (waveforms, labels) in enumerate(pbar):
        waveforms = waveforms.to(device)
        labels = labels.to(device)
        
        # Forward pass
        logits = model(waveforms)
        loss = F.cross_entropy(logits, labels, weight=class_weights)        
        loss = loss / accumulation_steps
        loss.backward()
        
        # Update weights only after accumulation_steps
        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
            optimizer.step()
            optimizer.zero_grad()

        # Statistics (use unscaled loss for reporting)
        total_loss += loss.item() * accumulation_steps
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
        
        # Update progress bar
        current_loss = total_loss / (batch_idx + 1)
        current_acc = correct / total
        pbar.set_postfix({'loss': f'{current_loss:.4f}', 'acc': f'{current_acc:.4f}'})

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total
    return avg_loss, accuracy


def validate(model, val_loader, device, epoch=None):
    """Validate model and compute detailed metrics"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    # For per-class metrics
    control_correct = 0
    control_total = 0
    dementia_correct = 0
    dementia_total = 0

    # For F1 score
    true_positives = 0
    false_positives = 0
    false_negatives = 0

    desc = f"Epoch {epoch} - Validation" if epoch is not None else "Validation"
    pbar = tqdm(val_loader, desc=desc, leave=False)

    with torch.no_grad():
        for waveforms, labels in pbar:
            waveforms = waveforms.to(device)
            labels = labels.to(device)
            
            logits = model(waveforms)
            loss = F.cross_entropy(logits, labels)
            predictions = torch.argmax(logits, dim=1)

            # Overall statistics
            total_loss += loss.item()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

            # Per-class accuracy
            for pred, label in zip(predictions, labels):
                if label == 0:  # Control
                    control_total += 1
                    if pred == label:
                        control_correct += 1
                else:  # Dementia
                    dementia_total += 1
                    if pred == label:
                        dementia_correct += 1

                # F1 score components
                if pred == 1 and label == 1:
                    true_positives += 1
                elif pred == 1 and label == 0:
                    false_positives += 1
                elif pred == 0 and label == 1:
                    false_negatives += 1
            
            # Update progress bar
            current_loss = total_loss / (pbar.n + 1)
            current_acc = correct / total
            pbar.set_postfix({'loss': f'{current_loss:.4f}', 'acc': f'{current_acc:.4f}'})

    avg_loss = total_loss / len(val_loader)
    accuracy = correct / total

    # Per-class accuracy
    control_acc = control_correct / control_total if control_total > 0 else 0
    dementia_acc = dementia_correct / dementia_total if dementia_total > 0 else 0

    # F1 score
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return avg_loss, accuracy, control_acc, dementia_acc, f1_score


In [ ]:
class AudioDataset(Dataset):
    """
    Dataset for loading audio files directly for XLSR fine-tuning
    
    Args:
        csv_path: Path to CSV file containing session IDs and labels
        audio_dir: Root directory for audio files (should contain Control/ and Dementia/ subdirs)
        max_length_sec: Maximum audio length in seconds
        sampling_rate: Target sampling rate
    """
    def __init__(self, csv_path, audio_dir, max_length_sec=60, sampling_rate=16000):
        self.csv_path = csv_path
        self.audio_dir = Path(audio_dir)
        self.max_length_sec = max_length_sec
        self.sampling_rate = sampling_rate
        self.max_samples = max_length_sec * sampling_rate
        
        # Load data from CSV
        self.data = []
        df = pd.read_csv(csv_path)
        
        for _, row in df.iterrows():
            session_id = row['session_id']
            label = int(row['ad'])
            
            # Construct audio file path based on label
            # 0 = Control, 1 = Dementia
            label_dir = "Dementia" if label == 1 else "Control"
            
            # Try different audio extensions
            found = False
            for ext in ['.wav', '.mp3', '.flac']:
                audio_path = self.audio_dir / label_dir / f"{session_id}{ext}"
                if audio_path.exists():
                    self.data.append({
                        'session_id': session_id,
                        'audio_path': audio_path,
                        'label': label
                    })
                    found = True
                    break
            
            if not found:
                print(f"Warning: Audio file not found for session {session_id} in {label_dir}/")
        
        print(f"Loaded {len(self.data)} audio files")
        control_count = sum(1 for d in self.data if d['label'] == 0)
        dementia_count = sum(1 for d in self.data if d['label'] == 1)
        print(f"Control: {control_count}, Dementia: {dementia_count}\n")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        audio_path = item['audio_path']
        label = item['label']
        
        # Load audio
        waveform, sr = torchaudio.load(audio_path)
        
        # Resample if necessary
        if sr != self.sampling_rate:
            resampler = torchaudio.transforms.Resample(sr, self.sampling_rate)
            waveform = resampler(waveform)
        
        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        
        # Squeeze to 1D
        waveform = waveform.squeeze(0)
        
        # Pad or truncate to max_samples
        if waveform.shape[0] > self.max_samples:
            waveform = waveform[:self.max_samples]
        elif waveform.shape[0] < self.max_samples:
            padding = self.max_samples - waveform.shape[0]
            waveform = F.pad(waveform, (0, padding))
        
        return waveform, label


def collate_audio_batch(batch):
    """
    Collate function for audio batch
    Returns:
        waveforms: (batch_size, time)
        labels: (batch_size,)
    """
    waveforms = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    
    waveforms = torch.stack(waveforms)
    labels = torch.tensor(labels, dtype=torch.long)
    
    return waveforms, labels


## Step 3: Create Data Loaders


In [ ]:
# Create datasets
train_dataset = AudioDataset(
    csv_path=TRAIN_CSV,
    audio_dir=RAW_AUDIO_DIR,
    max_length_sec=AUDIO_LENGTH_SEC,
    sampling_rate=SAMPLING_RATE
)

val_dataset = AudioDataset(
    csv_path=VAL_CSV,
    audio_dir=RAW_AUDIO_DIR,
    max_length_sec=AUDIO_LENGTH_SEC,
    sampling_rate=SAMPLING_RATE
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=FINETUNE_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_audio_batch,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=FINETUNE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_audio_batch,
    pin_memory=torch.cuda.is_available()
)


## Step 4: Define Fine-tuning Model

Create an end-to-end model that combines XLSR feature extraction with the AD classifier


In [ ]:
class XLSR_Finetune_Model(nn.Module):
    """
    End-to-end XLSR fine-tuning model for AD detection
    
    Combines:
    1. SSLModel (XLSR-53) - only last 3 transformer layers are trainable
    2. AD_XLSR_Model - AD classification head
    """
    def __init__(self, device, dropout=0.2, num_finetune_layers=FINETUNE_LAYERS_NUM):
        super().__init__()
        
        # XLSR feature extractor (freeze all first)
        self.ssl_model = SSLModel(device=device, freeze_xlsr=True)
        
        # Unfreeze only the last N transformer layers
        encoder_layers = self.ssl_model.model.encoder.layers
        total_layers = len(encoder_layers)
        for i, layer in enumerate(encoder_layers):
            if i >= total_layers - num_finetune_layers:
                for param in layer.parameters():
                    param.requires_grad = True
        
        self.ssl_model.model.train()
        print(f"XLSR: Training layers {total_layers - num_finetune_layers} to {total_layers - 1}")
        
        # AD classifier
        self.classifier = AD_XLSR_Model(dropout=dropout)
        
        self.device = device
    
    def forward(self, waveform, mask=None):
        """
        Args:
            waveform: (batch_size, time) - raw audio waveform
            mask: Optional attention mask (not used in current implementation)
        
        Returns:
            logits: (batch_size, 2) - AD classification logits
        """
        embedding, _ = self.ssl_model.extract_feat(waveform)
        batch_size, seq_len, _ = embedding.shape
        
        # Pad or truncate to XLSR_MAX_TIME_STEPS
        if seq_len > XLSR_MAX_TIME_STEPS:
            embedding = embedding[:, :XLSR_MAX_TIME_STEPS, :]
            mask = torch.ones(batch_size, XLSR_MAX_TIME_STEPS, device=self.device)
        elif seq_len < XLSR_MAX_TIME_STEPS:
            padding = torch.zeros(batch_size, XLSR_MAX_TIME_STEPS - seq_len, 1024, device=self.device)
            embedding = torch.cat([embedding, padding], dim=1)
            mask = torch.ones(batch_size, XLSR_MAX_TIME_STEPS, device=self.device)
            mask[:, seq_len:] = 0
        else:
            mask = torch.ones(batch_size, seq_len, device=self.device)
        
        # Classify
        logits = self.classifier(embedding, mask)
        
        return logits


## Step 5: Training Functions


In [ ]:
def train_finetune(seed, train_loader, val_loader, output_dir, device):
    """
    Fine-tuning pipeline for XLSR model
    
    Args:
        seed: Random seed
        train_loader: Training data loader
        val_loader: Validation data loader
        output_dir: Directory to save models
        device: Device to train on
    
    Returns:
        seed: The seed used
        best_metrics: Dictionary of best validation metrics
        training_history: Dictionary of training history
    """
    # Set random seed
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    np.random.seed(seed)

    # Create save directory
    seed_dir = Path(output_dir) / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    # Create model
    model = XLSR_Finetune_Model(device=device, dropout=XLSR_DROPOUT).to(device)
    
    # Create optimizer with differential learning rates
    xlsr_trainable_params = [p for p in model.ssl_model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW([
        {'params': xlsr_trainable_params, 'lr': XLSR_LR},
        {'params': model.classifier.parameters(), 'lr': CLASSIFIER_LR}
    ], weight_decay=WEIGHT_DECAY)

    # Set class weights from hyperparameters
    class_weights = torch.tensor([CLASS_WEIGHT_CONTROL, CLASS_WEIGHT_DEMENTIA], device=device)

    # Training history
    train_losses = []
    train_accs = []
    val_losses = []
    val_accs = []
    epochs_list = []

    # Early stopping
    best_val_acc = 0
    best_val_loss = float('inf')
    best_metrics = {}
    patience_counter = 0
    best_epoch = 0
    stopped_epoch = 0

    print(f"\n{'='*70}")
    print(f"Training with seed {seed}")
    print(f"Class weights: Control={class_weights[0]:.1f}, Dementia={class_weights[1]:.1f}")
    print(f"{'='*70}")

    # Training loop
    for epoch in range(FINETUNE_MAX_EPOCHS):
        # Train
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device, epoch=epoch+1, accumulation_steps=ACCUMULATION_STEPS, class_weights=class_weights)
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        # Validate
        val_loss, val_acc, control_acc, dementia_acc, f1 = validate(model, val_loader, device, epoch=epoch+1)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        epochs_list.append(epoch)

        # Print epoch results
        print(f"Epoch {epoch+1}/{FINETUNE_MAX_EPOCHS} - "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}% - "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}% | "
              f"Control: {control_acc*100:.2f}%, Dementia: {dementia_acc*100:.2f}%, F1: {f1:.4f}")

        # Save best model based on highest accuracy, then lowest loss
        is_best = False
        if val_acc > best_val_acc or (val_acc == best_val_acc and val_loss < best_val_loss):
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_metrics = {
                'val_acc': val_acc,
                'val_loss': val_loss,
                'control_acc': control_acc,
                'dementia_acc': dementia_acc,
                'f1_score': f1
            }
            patience_counter = 0
            is_best = True
            
            # Save fine-tuned XLSR model
            torch.save({
                'ssl_model_state_dict': model.ssl_model.state_dict(),
                'epoch': epoch,
                'best_val_acc': best_val_acc,
                'best_val_loss': best_val_loss,
            }, seed_dir / 'best_xlsr.pth')
            
            # Save classifier model
            torch.save({
                'classifier_state_dict': model.classifier.state_dict(),
                'epoch': epoch,
                'best_val_acc': best_val_acc,
                'best_val_loss': best_val_loss,
            }, seed_dir / 'best_classifier.pth')
            
            # Save optimizer state
            torch.save({
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'best_val_acc': best_val_acc,
                'best_val_loss': best_val_loss,
            }, seed_dir / 'optimizer.pth')
            
            print(f"  *** New best model saved! (Acc: {val_acc*100:.2f}%, Loss: {val_loss:.4f}) ***")
            print(f"      - Fine-tuned XLSR: best_xlsr.pth")
            print(f"      - Classifier: best_classifier.pth")
            print(f"      - Optimizer: optimizer.pth")
        else:
            patience_counter += 1

        # Early stopping check
        if patience_counter >= FINETUNE_PATIENCE:
            stopped_epoch = epoch + 1
            print(f"\nEarly stopping triggered after {FINETUNE_PATIENCE} epochs without improvement")
            break

    # Print final results
    print(f"\n{'='*70}")
    print(f"Training completed for seed {seed}")
    print(f"{'='*70}")
    print(f"Best Validation Accuracy: {best_metrics['val_acc']*100:.2f}%")
    print(f"Best Validation Loss: {best_metrics['val_loss']:.4f}")
    print(f"F1 Score: {best_metrics['f1_score']:.4f}")
    print(f"Control Accuracy: {best_metrics['control_acc']*100:.2f}%")
    print(f"Dementia Accuracy: {best_metrics['dementia_acc']*100:.2f}%")
    print(f"Best epoch: {best_epoch}")
    print(f"Stopped at epoch: {stopped_epoch if stopped_epoch > 0 else FINETUNE_MAX_EPOCHS}")
    print(f"{'='*70}\n")

    # Return training history along with best metrics
    training_history = {
        'epochs': epochs_list,
        'train_losses': train_losses,
        'train_accs': train_accs,
        'val_losses': val_losses,
        'val_accs': val_accs
    }

    return seed, best_metrics, training_history


## Step 7: Train with Multiple Seeds


In [ ]:
# Initialize results storage
all_results = {
    'seeds': [], 
    'val_accs': [], 
    'val_losses': [], 
    'control_accs': [], 
    'dementia_accs': [], 
    'f1_scores': []
}


## Random Seed 21

In [ ]:
seed, metrics, history = train_finetune(
    seed=21,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=MODEL_OUTPUT_DIR,
    device=device
)

all_results['seeds'].append(seed)
all_results['val_accs'].append(metrics['val_acc'])
all_results['val_losses'].append(metrics['val_loss'])
all_results['control_accs'].append(metrics['control_acc'])
all_results['dementia_accs'].append(metrics['dementia_acc'])
all_results['f1_scores'].append(metrics['f1_score'])

plot_training_curves(
    epochs=history['epochs'],
    train_loss=history['train_losses'],
    val_loss=history['val_losses'],
    train_acc=history['train_accs'],
    val_acc=history['val_accs'],
    title_prefix=f'Seed {seed} (Fine-tuning)'
)


In [ ]:
torch.cuda.empty_cache()

## 42

In [ ]:
seed, metrics, history = train_finetune(
    seed=42,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=MODEL_OUTPUT_DIR,
    device=device
)

all_results['seeds'].append(seed)
all_results['val_accs'].append(metrics['val_acc'])
all_results['val_losses'].append(metrics['val_loss'])
all_results['control_accs'].append(metrics['control_acc'])
all_results['dementia_accs'].append(metrics['dementia_acc'])
all_results['f1_scores'].append(metrics['f1_score'])

plot_training_curves(
    epochs=history['epochs'],
    train_loss=history['train_losses'],
    val_loss=history['val_losses'],
    train_acc=history['train_accs'],
    val_acc=history['val_accs'],
    title_prefix=f'Seed {seed} (Fine-tuning)'
)


In [ ]:
torch.cuda.empty_cache()

## 84

In [ ]:
seed, metrics, history = train_finetune(
    seed=84,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=MODEL_OUTPUT_DIR,
    device=device
)

all_results['seeds'].append(seed)
all_results['val_accs'].append(metrics['val_acc'])
all_results['val_losses'].append(metrics['val_loss'])
all_results['control_accs'].append(metrics['control_acc'])
all_results['dementia_accs'].append(metrics['dementia_acc'])
all_results['f1_scores'].append(metrics['f1_score'])

plot_training_curves(
    epochs=history['epochs'],
    train_loss=history['train_losses'],
    val_loss=history['val_losses'],
    train_acc=history['train_accs'],
    val_acc=history['val_accs'],
    title_prefix=f'Seed {seed} (Fine-tuning)'
)


In [ ]:
import torch
torch.cuda.empty_cache()

## 168

In [ ]:
seed, metrics, history = train_finetune(
    seed=168,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=MODEL_OUTPUT_DIR,
    device=device
)

all_results['seeds'].append(seed)
all_results['val_accs'].append(metrics['val_acc'])
all_results['val_losses'].append(metrics['val_loss'])
all_results['control_accs'].append(metrics['control_acc'])
all_results['dementia_accs'].append(metrics['dementia_acc'])
all_results['f1_scores'].append(metrics['f1_score'])

plot_training_curves(
    epochs=history['epochs'],
    train_loss=history['train_losses'],
    val_loss=history['val_losses'],
    train_acc=history['train_accs'],
    val_acc=history['val_accs'],
    title_prefix=f'Seed {seed} (Fine-tuning)'
)


In [ ]:
torch.cuda.empty_cache()

## 336

In [ ]:
seed, metrics, history = train_finetune(
    seed=336,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=MODEL_OUTPUT_DIR,
    device=device
)

all_results['seeds'].append(seed)
all_results['val_accs'].append(metrics['val_acc'])
all_results['val_losses'].append(metrics['val_loss'])
all_results['control_accs'].append(metrics['control_acc'])
all_results['dementia_accs'].append(metrics['dementia_acc'])
all_results['f1_scores'].append(metrics['f1_score'])

plot_training_curves(
    epochs=history['epochs'],
    train_loss=history['train_losses'],
    val_loss=history['val_losses'],
    train_acc=history['train_accs'],
    val_acc=history['val_accs'],
    title_prefix=f'Seed {seed} (Fine-tuning)'
)


In [ ]:
torch.cuda.empty_cache()

## Step 6: Summary of Results

In [ ]:
seeds = all_results['seeds']
val_accs = [acc*100 for acc in all_results['val_accs']]
val_losses = all_results['val_losses']
control_accs = [acc*100 for acc in all_results['control_accs']]
dementia_accs = [acc*100 for acc in all_results['dementia_accs']]
f1_scores = all_results['f1_scores']

# Calculate statistics (mean and std)
mean_acc = np.mean(val_accs)
std_acc = np.std(val_accs, ddof=1)

mean_loss = np.mean(val_losses)
std_loss = np.std(val_losses, ddof=1)

mean_control_acc = np.mean(control_accs)
std_control_acc = np.std(control_accs, ddof=1)

mean_dementia_acc = np.mean(dementia_accs)
std_dementia_acc = np.std(dementia_accs, ddof=1)

mean_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores, ddof=1)

print(f"\nMean Validation Accuracy: {mean_acc:.2f}% ± {std_acc:.2f}%")
print(f"Mean Validation Loss: {mean_loss:.4f} ± {std_loss:.4f}")
print(f"Mean Control Accuracy: {mean_control_acc:.2f}% ± {std_control_acc:.2f}%")
print(f"Mean Dementia Accuracy: {mean_dementia_acc:.2f}% ± {std_dementia_acc:.2f}%")
print(f"Mean F1 Score: {mean_f1:.4f} ± {std_f1:.4f}")

print(f"\nDetailed Results:")
for i, seed in enumerate(seeds):
    print(f"Seed {seed}: Acc={val_accs[i]:.2f}%, Loss={val_losses[i]:.4f}, "
          f"Control Acc={control_accs[i]:.2f}%, Dementia Acc={dementia_accs[i]:.2f}%, F1={f1_scores[i]:.4f}")

## Step 7: Test Best Model on Lu Dataset


In [ ]:
# Find the best model: highest accuracy, then lowest loss if tied
max_acc = max(all_results['val_accs'])
# Find all indices with max accuracy
max_acc_indices = [i for i, acc in enumerate(all_results['val_accs']) if acc == max_acc]

# If multiple models have the same max accuracy, choose the one with lowest loss
if len(max_acc_indices) > 1:
    best_idx = min(max_acc_indices, key=lambda i: all_results['val_losses'][i])
    print(f"Multiple models with accuracy {max_acc*100:.2f}%, selecting one with lowest loss")
else:
    best_idx = max_acc_indices[0]

BEST_SEED = all_results['seeds'][best_idx]
BEST_VAL_ACC = all_results['val_accs'][best_idx] * 100
BEST_VAL_LOSS = all_results['val_losses'][best_idx]
BEST_F1 = all_results['f1_scores'][best_idx]
BEST_CONTROL_ACC = all_results['control_accs'][best_idx] * 100
BEST_DEMENTIA_ACC = all_results['dementia_accs'][best_idx] * 100

print(f"\nBest Model: Seed {BEST_SEED}")
print(f"Validation Accuracy: {BEST_VAL_ACC:.2f}%")
print(f"Validation Loss: {BEST_VAL_LOSS:.4f}")
print(f"F1 Score: {BEST_F1:.4f}")
print(f"Control Accuracy: {BEST_CONTROL_ACC:.2f}%")
print(f"Dementia Accuracy: {BEST_DEMENTIA_ACC:.2f}%")

BEST_MODEL_DIR = MODEL_OUTPUT_DIR / f"seed_{BEST_SEED}"
BEST_XLSR_PATH = BEST_MODEL_DIR / "best_xlsr.pth"
BEST_CLASSIFIER_PATH = BEST_MODEL_DIR / "best_classifier.pth"
print(f"\nModel Directory: {BEST_MODEL_DIR}")
print(f"XLSR Path: {BEST_XLSR_PATH}")
print(f"Classifier Path: {BEST_CLASSIFIER_PATH}")


In [ ]:
# Lu dataset configuration
TEST_DATASET = "Lu"
LU_AUDIO_DIR = PROJECT_ROOT / "data/raw/Lu"
LU_CSV_PATH = PROJECT_ROOT / f"{TEST_DATASET}-xlsr-test.csv"

# Collect audio files from Lu dataset and create CSV
test_samples = []

for label_dir in ["Control", "Dementia"]:
    label_audio_dir = LU_AUDIO_DIR / label_dir
    if not label_audio_dir.exists():
        print(f"Warning: Directory does not exist: {label_audio_dir}")
        continue
    
    label = 0 if label_dir == "Control" else 1
    
    # Find all audio files
    audio_files = list(label_audio_dir.glob("*.wav")) + list(label_audio_dir.glob("*.mp3")) + list(label_audio_dir.glob("*.flac"))
    print(f"Found {len(audio_files)} audio files in {label_dir}/")
    
    for audio_file in sorted(audio_files):
        session_id = audio_file.stem
        test_samples.append({
            "session_id": session_id,
            "ad": label
        })

# Create CSV for test dataset
test_df = pd.DataFrame(test_samples)
test_df.to_csv(LU_CSV_PATH, index=False)

print(f"\nLu Dataset Summary:")
print(f"Total samples: {len(test_samples)}")
print(f"Control: {sum(1 for s in test_samples if s['ad'] == 0)}")
print(f"Dementia: {sum(1 for s in test_samples if s['ad'] == 1)}")
print(f"CSV saved to: {LU_CSV_PATH}")

In [ ]:
# Create test dataset and dataloader for Lu
lu_test_dataset = AudioDataset(
    csv_path=LU_CSV_PATH,
    audio_dir=LU_AUDIO_DIR,
    max_length_sec=AUDIO_LENGTH_SEC,
    sampling_rate=SAMPLING_RATE
)

lu_test_loader = DataLoader(
    lu_test_dataset,
    batch_size=FINETUNE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_audio_batch,
    pin_memory=torch.cuda.is_available()
)

print(f"Lu test batches: {len(lu_test_loader)}")

In [ ]:
# Load the best fine-tuned model
print(f"Loading best model from seed {BEST_SEED}...")
print(f"  XLSR checkpoint: {BEST_XLSR_PATH}")
print(f"  Classifier checkpoint: {BEST_CLASSIFIER_PATH}")

# Create the fine-tuning model (same architecture as training)
test_model = XLSR_Finetune_Model(device=device, dropout=XLSR_DROPOUT)

# Load fine-tuned XLSR weights
xlsr_checkpoint = torch.load(BEST_XLSR_PATH, map_location=device, weights_only=False)
test_model.ssl_model.load_state_dict(xlsr_checkpoint['ssl_model_state_dict'])
print(f"  Loaded XLSR from epoch {xlsr_checkpoint['epoch']+1}, val_acc: {xlsr_checkpoint['best_val_acc']*100:.2f}%")

# Load classifier weights
classifier_checkpoint = torch.load(BEST_CLASSIFIER_PATH, map_location=device, weights_only=False)
test_model.classifier.load_state_dict(classifier_checkpoint['classifier_state_dict'])
print(f"  Loaded classifier from epoch {classifier_checkpoint['epoch']+1}")

test_model = test_model.to(device)
test_model.eval()

print(f"\nModel loaded successfully!")

In [ ]:
# Test on Lu dataset
from sklearn.metrics import accuracy_score, f1_score as sklearn_f1_score

# Perform predictions
all_preds = []
all_labels = []
all_probs = []

test_model.eval()
with torch.no_grad():
    for waveforms, labels in tqdm(lu_test_loader, desc=f"Testing on {TEST_DATASET}"):
        waveforms = waveforms.to(device)
        labels = labels.to(device)
        
        logits = test_model(waveforms)
        probs = F.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Calculate overall metrics
lu_accuracy = accuracy_score(all_labels, all_preds)
lu_f1 = sklearn_f1_score(all_labels, all_preds, pos_label=1)

# Calculate per-class accuracy
control_mask = all_labels == 0
dementia_mask = all_labels == 1

lu_control_acc = np.mean(all_preds[control_mask] == all_labels[control_mask]) if control_mask.sum() > 0 else 0
lu_dementia_acc = np.mean(all_preds[dementia_mask] == all_labels[dementia_mask]) if dementia_mask.sum() > 0 else 0

print(f"\n{'='*60}")
print(f"Test Results on {TEST_DATASET} Dataset")
print(f"{'='*60}")
print(f"Overall Accuracy: {lu_accuracy*100:.2f}%")
print(f"F1 Score: {lu_f1:.4f}")
print(f"Control Accuracy: {lu_control_acc*100:.2f}%")
print(f"Dementia Accuracy: {lu_dementia_acc*100:.2f}%")
print(f"{'='*60}")

In [ ]:
# Performance Comparison Visualization
import matplotlib.pyplot as plt

# Set font properties
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# Prepare data for visualization
dataset_names = [f"{DATASET_NAME}\n(Validation)", f"{TEST_DATASET}\n(Test)"]
accuracies = [BEST_VAL_ACC, lu_accuracy * 100]

# Set distinct colors for each dataset
PITT_COLOR = '#27F5EE'   # Blue for Pitt
LU_COLOR = '#F5A623'     # Red for Lu
colors = [PITT_COLOR, LU_COLOR]

# Create bar chart
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(dataset_names, accuracies, color=colors, edgecolor='black', linewidth=1.5, width=0.5)

# Set labels and title
ax.set_ylabel('Accuracy (%)', fontsize=14, fontweight='bold')
ax.set_title(f'{DATASET_NAME}-Trained Model Performance Comparison', fontsize=16, fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis='y')
ax.set_axisbelow(True)

# Add value labels on top of bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{acc:.2f}%',
            ha='center', va='bottom',
            fontweight='bold', fontsize=14, color='black')

# Add legend
ax.legend(bars, [f'{DATASET_NAME} (Val)', f'{TEST_DATASET} (Test)'], 
          loc='upper right', fontsize=12)

plt.tight_layout()
plt.show()

# Print comprehensive summary
print(f"\n{'='*70}")
print(f"FINAL COMPARISON: {DATASET_NAME} vs {TEST_DATASET}")
print(f"{'='*70}")

print(f"\n{DATASET_NAME} Dataset (Validation - Best Seed {BEST_SEED}):")
print(f"  Overall Accuracy: {BEST_VAL_ACC:.2f}%")
print(f"  F1 Score: {BEST_F1:.4f}")
print(f"  Control Accuracy: {BEST_CONTROL_ACC:.2f}%")
print(f"  Dementia Accuracy: {BEST_DEMENTIA_ACC:.2f}%")

print(f"\n{TEST_DATASET} Dataset (Test):")
print(f"  Overall Accuracy: {lu_accuracy*100:.2f}%")
print(f"  F1 Score: {lu_f1:.4f}")
print(f"  Control Accuracy: {lu_control_acc*100:.2f}%")
print(f"  Dementia Accuracy: {lu_dementia_acc*100:.2f}%")

perf_diff = lu_accuracy*100 - BEST_VAL_ACC
print(f"\nPerformance Difference: {perf_diff:+.2f}%")